# TabNet: A Tabular Neural Network for Ground Motion Prediction
- Dataset: dataset/Updated_NGA_West2_Flatfile_RotD50_d050_public_version.xlsx
- Model: TabNet (pytorch-tabnet) — attention-based tabular neural network

### Tasks:
- Random search for best TabNet hyperparameters (N_d, N_steps, momentum, lambda_sparse, lr, batch_size)
- Training TabNet with the best hyperparameters
- Evaluating the performance of TabNet
- Parametric Analysis
- Bin residual plots
- TabNet native attention-based feature importance
- SHAP analysis and feature importance

### Inputs:
- Earth Magnitude
- Joyner-Boore Dist. (km)
- Mechanism Based on Rake Angle 
- Vs30 (m/s) selected for analysis
- Log10 of Vs30 (m/s)
- Log10 of Joyner-Boore Dist. (km)

### Outputs:
- Log10(PGV (cm/sec))
- Log10(PGA (g))
- Log10 of Spectral Acceleration from 0.01s to 4s: (T0.010S - T4.00S)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import itertools
import torch
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os

print(f"✓ PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device}")
if device.type == 'cuda':
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU detected — on Kaggle: Session ▸ Accelerator ▸ GPU T4 x2")


✓ PyTorch version: 2.6.0+cu124
✓ Device: cuda
✓ GPU: NVIDIA GeForce RTX 3050 Laptop GPU
✓ VRAM: 4.0 GB


In [2]:
import torch
import sys

if not torch.cuda.is_available():
    print("❌ No GPU → exiting")
    sys.exit(1)

gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name)

# 🚨 Reject bad GPU
if "P100" in gpu_name:
    print("❌ P100 detected → incompatible → exiting to retry")
    sys.exit(1)

print("✅ Good GPU detected → continuing")

GPU: NVIDIA GeForce RTX 3050 Laptop GPU
✅ Good GPU detected → continuing


In [3]:
import importlib.util
import subprocess
import sys

# Dictionary mapping the 'import' name to the 'pip install' name
dependencies = {
    'shap':          'shap',
    'calamine':      'python-calamine',
    'pytorch_tabnet': 'pytorch-tabnet',
}

for module, package in dependencies.items():
    if importlib.util.find_spec(module) is None:
        print(f"📦 {package} not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    else:
        print(f"✅ {package} is already installed.")

from pytorch_tabnet.tab_model import TabNetRegressor
print("✅ TabNetRegressor ready")

✅ shap is already installed.
📦 python-calamine not found. Installing...
✅ pytorch-tabnet is already installed.
✅ TabNetRegressor ready


In [4]:
import glob

# 1. Exact paths for Kaggle cloud vs your local WSL environment
FILENAME    = 'Updated_NGA_West2_Flatfile_RotD50_d050_public_version.xlsx'
kaggle_path = f'/kaggle/input/datasets/premsuggu11/nga-west2-seismic-data/{FILENAME}'
local_path  = f'dataset/{FILENAME}'
# 2. Automatically detect the environment
if os.path.exists('/kaggle/input'):
    print("☁️ Running on Kaggle!")

    if os.path.exists(kaggle_path):
        # Dataset attached with expected slug
        file_path = kaggle_path
        print(f"✅ Found dataset at expected path: {file_path}")
    else:
        # Dataset was attached but with a different slug — search recursively
        matches = [m for m in glob.glob('/kaggle/input/**/*.xlsx', recursive=True)
                   if FILENAME in m]
        if matches:
            file_path = matches[0]
            print(f"✅ Found dataset (auto-discovered): {file_path}")
        else:
            # Dataset not attached at all — show a helpful error
            print("\n❌ Dataset NOT found under /kaggle/input/")
            print("   Available inputs:", os.listdir('/kaggle/input') or "(empty)")
            raise FileNotFoundError(
                "\n\nTo fix this, attach the dataset to your notebook:\n"
                "  1. Open this notebook on Kaggle\n"
                "  2. Right panel → 'Input' tab → '+ Add Input'\n"
                "  3. Search for 'nga-west2-seismic-data' (or 'premsuggu11/nga-west2-seismic-data')\n"
                "  4. Click 'Add' then re-run all cells\n"
            )
else:
    print("💻 Running locally! Using local dataset path.")
    file_path = local_path

# 3. Load the data using the calamine engine
try:
    test_df = pd.read_excel(file_path, nrows=1, engine='calamine')
    print("✅ Columns loaded successfully:", len(test_df.columns), "columns")
except FileNotFoundError:
    print(f"❌ ERROR: Could not find the file at {file_path}")

💻 Running locally! Using local dataset path.
✅ Columns loaded successfully: 274 columns


In [5]:
len(test_df.columns)

274

In [6]:
# Helper function to convert period column name to float
def period_to_float(col_name):
    """Convert column name like T0.010S to 0.010"""
    return float(col_name[1:-1])

# Get all column names first
print("Reading column names...")
df_temp = pd.read_excel(file_path, engine='calamine', nrows=0)
all_cols = df_temp.columns.tolist()

# Find period columns <= 4.0 seconds only
period_cols = [col for col in all_cols if col.startswith('T') and col.endswith('S')]
selected_period_cols = [col for col in period_cols if period_to_float(col) <= 4.0]
selected_period_cols_sorted = sorted(selected_period_cols, key=period_to_float)

# Define columns to load
selected_columns = [
    'Earthquake Magnitude',
    'Joyner-Boore Dist. (km)',
    'Vs30 (m/s) selected for analysis',
    'Mechanism Based on Rake Angle',
    'PGA (g)',
    'PGV (cm/sec)'
] + selected_period_cols_sorted

# Define dtypes for faster loading
dtype_dict = {
    'Earthquake Magnitude': 'float32',
    'Joyner-Boore Dist. (km)': 'float32',
    'Vs30 (m/s) selected for analysis': 'float32',
    'Mechanism Based on Rake Angle': 'int8',
    'PGA (g)': 'float32',
    'PGV (cm/sec)': 'float32'
}
for col in selected_period_cols_sorted:
    dtype_dict[col] = 'float32'

print(f"Loading {len(selected_columns)} columns (6 base features + {len(selected_period_cols_sorted)} periods)...")
print(f"Period range: {period_to_float(selected_period_cols_sorted[0]):.3f}s to {period_to_float(selected_period_cols_sorted[-1]):.3f}s")

# Load data
df = pd.read_excel(file_path, engine='calamine', usecols=selected_columns, dtype=dtype_dict)

print(f"✓ Loaded data shape: {df.shape}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


Reading column names...
Loading 96 columns (6 base features + 90 periods)...
Period range: 0.010s to 4.000s
✓ Loaded data shape: (21540, 96)
  Memory usage: 7.83 MB


In [7]:
# Data Cleaning: Remove rows with negative, NaN, or Inf values
print("=" * 70)
print("DATA CLEANING")
print("=" * 70)
print(f"\nOriginal shape: {df.shape}")

cols_to_check = [
    'Earthquake Magnitude',
    'Joyner-Boore Dist. (km)',
    'Vs30 (m/s) selected for analysis',
    'Mechanism Based on Rake Angle',
    'PGA (g)',
    'PGV (cm/sec)'
] + selected_period_cols_sorted

# Step 1: Remove rows with any negative values
print("\nRemoving rows with negative values...")
before = len(df)
for col in cols_to_check:
    df = df[df[col] >= 0]
print(f"  After removing negatives: {len(df):,} rows (removed {before - len(df):,})")

# Step 2: Drop NaN / Inf rows
before = len(df)
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=cols_to_check)
print(f"  After dropping NaN/Inf:   {len(df):,} rows (removed {before - len(df):,})")

df = df.reset_index(drop=True)

print(f"\n✓ Final cleaned shape: {df.shape}")
print(f"\nData ranges:")
print(f"  Earthquake Magnitude:          {df['Earthquake Magnitude'].min():.2f} – {df['Earthquake Magnitude'].max():.2f}")
print(f"  Joyner-Boore Dist. (km):       {df['Joyner-Boore Dist. (km)'].min():.3f} – {df['Joyner-Boore Dist. (km)'].max():.2f}")
print(f"  Vs30 (m/s):                    {df['Vs30 (m/s) selected for analysis'].min():.1f} – {df['Vs30 (m/s) selected for analysis'].max():.1f}")
print(f"  Mechanism Based on Rake Angle: {sorted(df['Mechanism Based on Rake Angle'].unique())}")
print(f"  PGA (g):                       {df['PGA (g)'].min():.6f} – {df['PGA (g)'].max():.4f}")
print(f"  PGV (cm/sec):                  {df['PGV (cm/sec)'].min():.6f} – {df['PGV (cm/sec)'].max():.4f}")
print("=" * 70)


DATA CLEANING

Original shape: (21540, 96)

Removing rows with negative values...
  After removing negatives: 21,293 rows (removed 247)
  After dropping NaN/Inf:   21,293 rows (removed 0)

✓ Final cleaned shape: (21293, 96)

Data ranges:
  Earthquake Magnitude:          2.99 – 7.90
  Joyner-Boore Dist. (km):       0.000 – 1532.66
  Vs30 (m/s):                    89.3 – 2100.0
  Mechanism Based on Rake Angle: [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4)]
  PGA (g):                       0.000000 – 1.7683
  PGV (cm/sec):                  0.000007 – 256.6200


In [8]:
# Log-transform features and targets, then define input/output columns
print("=" * 70)
print("PREPROCESSING")
print("=" * 70)

df = df.reset_index(drop=True)

# Log10-transformed inputs
df['log10_Rjb']  = np.log10(df['Joyner-Boore Dist. (km)'])
df['log10_Vs30'] = np.log10(df['Vs30 (m/s) selected for analysis'])

# Log10-transformed targets
df['log10_PGA'] = np.log10(df['PGA (g)'])
df['log10_PGV'] = np.log10(df['PGV (cm/sec)'])

print("Creating log10 of spectral periods...")
for col in tqdm(selected_period_cols_sorted, desc="Log-transforming Sa"):
    df[f'log10_{col}'] = np.log10(df[col])

# Column lists used downstream
input_feature_cols = [
    'Earthquake Magnitude',
    'Joyner-Boore Dist. (km)',
    'Mechanism Based on Rake Angle',
    'Vs30 (m/s) selected for analysis',
    'log10_Vs30',
    'log10_Rjb'
]
output_target_cols = (
    ['log10_PGA', 'log10_PGV'] +
    [f'log10_{col}' for col in selected_period_cols_sorted]
)

# Sanity check: no inf/nan in transformed columns
all_transformed = ['log10_Rjb', 'log10_Vs30', 'log10_PGA', 'log10_PGV'] + \
                  [f'log10_{col}' for col in selected_period_cols_sorted]
invalid = df[all_transformed].replace([np.inf, -np.inf], np.nan).isna().any(axis=1).sum()
if invalid:
    print(f"  ⚠ Dropping {invalid} rows with inf/nan after log transform")
    df = df[~df[all_transformed].replace([np.inf, -np.inf], np.nan).isna().any(axis=1)]
    df = df.reset_index(drop=True)
else:
    print("  ✓ All log-transformed values are finite")

print(f"\nFinal dataset: {len(df):,} samples")
print(f"\nInputs  ({len(input_feature_cols)}): {input_feature_cols}")
print(f"Outputs ({len(output_target_cols)}): log10_PGA, log10_PGV + {len(selected_period_cols_sorted)} Sa periods")
print("=" * 70)


/home/prem/miniconda3/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


PREPROCESSING
Creating log10 of spectral periods...


Log-transforming Sa: 100%|██████████| 90/90 [00:00<00:00, 715.81it/s]

  ⚠ Dropping 63 rows with inf/nan after log transform



Final dataset: 21,230 samples

Inputs  (6): ['Earthquake Magnitude', 'Joyner-Boore Dist. (km)', 'Mechanism Based on Rake Angle', 'Vs30 (m/s) selected for analysis', 'log10_Vs30', 'log10_Rjb']
Outputs (92): log10_PGA, log10_PGV + 90 Sa periods


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — all tuneable variables in one place
# ══════════════════════════════════════════════════════════════════════════════

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = '/kaggle/working/results' if os.path.exists('/kaggle/input') else 'results/tabnet'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Global seed ───────────────────────────────────────────────────────────────
RANDOM_SEED = 100
np.random.seed(RANDOM_SEED)

# ── Hyperparameter search ─────────────────────────────────────────────────────
NUM_CONFIGS     = 300    # number of random configs to evaluate (raise for wider sweep)
MAX_EPOCHS_SRCH = 30   # max epochs per search trial
PATIENCE_SRCH   = 20   # early-stopping patience during search
VAL_FRAC        = 0.15 # fraction of train set held out as validation

# ── Final model training ──────────────────────────────────────────────────────
MAX_EPOCHS = 300
PATIENCE   = 50

# ── Search space (n_a is always set equal to n_d) ─────────────────────────────
SWEEP_CONFIG = {
    'n_d':           [8, 16, 32, 64, 128],            # decision-step embedding width
    'n_steps':       [3, 5, 7, 10],              # number of sequential attention steps
    'momentum':      [0.01, 0.02, 0.03, 0.05],         # batch-norm momentum
    'lambda_sparse': [0.0, 1e-4, 1e-3, 1e-2],   # sparsity regularisation strength
    'lr':            [1e-4, 5e-4, 1e-3, 3e-3],  # Adam learning rate
    'batch_size':    [256, 512, 1024],            # mini-batch size (GPU-friendly)
}

BEST_CONFIG = None
# ── Optional: skip search by supplying the best config directly ───────────────
# Set BEST_CONFIG = None  →  runs the random search (NUM_CONFIGS trials)
# Set BEST_CONFIG = {...} →  skips the search entirely and uses this config
# BEST_CONFIG = {
#     'n_d':           32,
#     'n_steps':       3,
#     'momentum':      0.01,
#     'lambda_sparse': 1e-3,
#     'lr':            3e-3,
#     'batch_size':    256,
# }

total_grid = 1
for v in SWEEP_CONFIG.values():
    total_grid *= len(v)

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"  Output dir      : {OUTPUT_DIR}")
print(f"  Device          : {device}")
print(f"  BEST_CONFIG     : {'provided (search will be skipped)' if BEST_CONFIG else 'None (search will run)'}")
print(f"  NUM_CONFIGS     : {NUM_CONFIGS}  (out of {total_grid:,} possible)")
print(f"  MAX_EPOCHS_SRCH : {MAX_EPOCHS_SRCH}  (patience={PATIENCE_SRCH})")
print(f"  MAX_EPOCHS      : {MAX_EPOCHS}  (patience={PATIENCE})")
print(f"  VAL_FRAC        : {VAL_FRAC}")
print(f"\n  Search space:")
for k, v in SWEEP_CONFIG.items():
    print(f"    {k:<16}: {v}")
print("=" * 60)


CONFIGURATION
  Output dir      : results/tabnet
  Device          : cuda
  BEST_CONFIG     : None (search will run)
  NUM_CONFIGS     : 300  (out of 3,840 possible)
  MAX_EPOCHS_SRCH : 30  (patience=20)
  MAX_EPOCHS      : 300  (patience=50)
  VAL_FRAC        : 0.15

  Search space:
    n_d             : [8, 16, 32, 64, 128]
    n_steps         : [3, 5, 7, 10]
    momentum        : [0.01, 0.02, 0.03, 0.05]
    lambda_sparse   : [0.0, 0.0001, 0.001, 0.01]
    lr              : [0.0001, 0.0005, 0.001, 0.003]
    batch_size      : [256, 512, 1024]


In [10]:
# Train / Test split + StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[input_feature_cols].values.astype('float32')
y = df[output_target_cols].values.astype('float32')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)

input_dim  = len(input_feature_cols)
output_dim = len(output_target_cols)

# ── Val split carved from train (reused in search and final training) ─────────
n_val_split = int(len(X_train_scaled) * VAL_FRAC)
X_srch_tr   = X_train_scaled[n_val_split:]
y_srch_tr   = y_train_scaled[n_val_split:]
X_srch_val  = X_train_scaled[:n_val_split]
y_srch_val  = y_train_scaled[:n_val_split]

sa_periods = np.array([period_to_float(col) for col in selected_period_cols_sorted])

print(f"Train total  : {X_train_scaled.shape[0]:,} samples")
print(f"Test         : {X_test_scaled.shape[0]:,} samples")
print(f"  Search sub-train : {X_srch_tr.shape[0]:,}  |  val : {X_srch_val.shape[0]:,}")
print(f"X shape: {X_train_scaled.shape}  |  y shape: {y_train_scaled.shape}")
print(f"SA periods: {len(sa_periods)} periods from {sa_periods[0]:.3f}s to {sa_periods[-1]:.3f}s")


Train total  : 16,984 samples
Test         : 4,246 samples
  Search sub-train : 14,437  |  val : 2,547
X shape: (16984, 6)  |  y shape: (16984, 92)
SA periods: 90 periods from 0.010s to 4.000s


In [ ]:
# ── Phase 2: Random Hyperparameter Search ─────────────────────────────────────
print("=" * 70)
print("PHASE 2: RANDOM HYPERPARAMETER SEARCH")
print("=" * 70)

param_names  = list(SWEEP_CONFIG.keys())
param_values = list(SWEEP_CONFIG.values())
all_combos   = list(itertools.product(*param_values))

if BEST_CONFIG is not None:
    # ── Skip search: use provided config ──────────────────────────────────────
    print("⏭️  BEST_CONFIG provided — skipping search.\n")
    best_config  = BEST_CONFIG
    best_val_mse = float('nan')
    search_results = [{**best_config, 'val_mse': best_val_mse, 'epochs': 0}]
    print("  Using config:")
    for k, v in best_config.items():
        print(f"    {k:<16}: {v}")
else:
    # ── Run random search ─────────────────────────────────────────────────────
    print(f"Sampling {NUM_CONFIGS} random configs from {total_grid:,} combinations")
    print(f"Per trial: max_epochs={MAX_EPOCHS_SRCH}, patience={PATIENCE_SRCH}\n")

    rng          = np.random.default_rng(RANDOM_SEED)
    sampled_idx  = rng.choice(len(all_combos), size=NUM_CONFIGS, replace=False)
    sampled_cfgs = [dict(zip(param_names, all_combos[i])) for i in sampled_idx]

    search_results = []
    for i, cfg in enumerate(sampled_cfgs):
        model = TabNetRegressor(
            n_d              = cfg['n_d'],
            n_a              = cfg['n_d'],
            n_steps          = cfg['n_steps'],
            momentum         = cfg['momentum'],
            lambda_sparse    = cfg['lambda_sparse'],
            optimizer_params = {'lr': cfg['lr']},
            seed             = RANDOM_SEED,
            device_name      = 'auto',
            verbose          = 0,
        )
        model.fit(
            X_srch_tr, y_srch_tr,
            eval_set           = [(X_srch_val, y_srch_val)],
            eval_metric        = ['mse'],
            max_epochs         = MAX_EPOCHS_SRCH,
            patience           = PATIENCE_SRCH,
            batch_size         = cfg['batch_size'],
            virtual_batch_size = max(8, cfg['batch_size'] // 4),
            drop_last          = False,
        )
        val_mse     = min(model.history['val_0_mse'])
        epochs_done = len(model.history['loss'])
        search_results.append({**cfg, 'val_mse': val_mse, 'epochs': epochs_done})
        print(f"  [{i+1:2d}/{NUM_CONFIGS}] n_d={cfg['n_d']:2d}, steps={cfg['n_steps']:2d}, "
              f"lr={cfg['lr']:.0e}, bs={cfg['batch_size']:4d}, "
              f"λ={cfg['lambda_sparse']:.0e}  →  val_mse={val_mse:.6f}  (ep={epochs_done})")

    search_results.sort(key=lambda r: r['val_mse'])
    best_config  = {k: search_results[0][k] for k in param_names}
    best_val_mse = search_results[0]['val_mse']

print(f"\n{'='*70}")
print(f"✓ Best config  (val_mse={best_val_mse}):")
for k, v in best_config.items():
    print(f"    {k:<16}: {v}")
print(f"{'='*70}")


PHASE 2: RANDOM HYPERPARAMETER SEARCH
Sampling 300 random configs from 3,840 combinations
Per trial: max_epochs=30, patience=20



In [ ]:
# ── Search Results: val_mse per config (sorted best → worst) ─────────────────
n_configs_evaluated = len(search_results)
val_mse_sorted = [r['val_mse'] for r in search_results]

if BEST_CONFIG is not None:
    # Search was skipped — just print a summary, no bar chart
    print("⏭️  Search was skipped (BEST_CONFIG was provided).")
    print(f"\n  Config in use:")
    for k, v in best_config.items():
        print(f"    {k:<16}: {v}")
else:
    fig, ax = plt.subplots(figsize=(max(10, n_configs_evaluated // 2), 5))
    bar_colors = ['#27AE60'] + ['#3498DB'] * (n_configs_evaluated - 1)
    ax.bar(range(n_configs_evaluated), val_mse_sorted, color=bar_colors,
           edgecolor='white', linewidth=0.4)
    ax.bar(0, val_mse_sorted[0], color='#27AE60',
           edgecolor='gold', linewidth=2.5,
           label=(f'Best  val_mse={best_val_mse:.6f}\n'
                  f'n_d={best_config["n_d"]}, n_steps={best_config["n_steps"]}, '
                  f'lr={best_config["lr"]:.0e}, bs={best_config["batch_size"]}, '
                  f'λ={best_config["lambda_sparse"]:.0e}'))
    ax.set_xticks(range(n_configs_evaluated))
    ax.set_xticklabels([str(i + 1) for i in range(n_configs_evaluated)], fontsize=8)
    ax.set_xlabel('Config rank (sorted by val MSE, best = 1)',
                  fontsize=12, fontweight='bold')
    ax.set_ylabel('Validation MSE', fontsize=12, fontweight='bold')
    ax.set_title(f'Random Hyperparameter Search — {n_configs_evaluated} Configs Evaluated',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, framealpha=0.9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/search_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {OUTPUT_DIR}/search_results.png")

    # ── Top-5 summary table ───────────────────────────────────────────────────
    top_n = min(5, n_configs_evaluated)
    print(f"\nTop-{top_n} configs:")
    print(f"  {'Rank':<5} {'n_d':>4} {'n_steps':>7} {'lr':>8} {'batch':>6} "
          f"{'λ_sparse':>10} {'momentum':>10} {'val_mse':>12}")
    print("  " + "-" * 65)
    for rank, r in enumerate(search_results[:top_n], 1):
        print(f"  {rank:<5} {r['n_d']:>4} {r['n_steps']:>7} {r['lr']:>8.0e} "
              f"{r['batch_size']:>6} {r['lambda_sparse']:>10.0e} "
              f"{r['momentum']:>10.3f} {r['val_mse']:>12.6f}")


In [ ]:
# ── Phase 3: Final Model Training ─────────────────────────────────────────────
print("=" * 70)
print("PHASE 3: FINAL TRAINING")
print("=" * 70)
print(f"  n_d={best_config['n_d']}, n_a={best_config['n_d']}, "
      f"n_steps={best_config['n_steps']}")
print(f"  lr={best_config['lr']:.0e}, batch_size={best_config['batch_size']}, "
      f"momentum={best_config['momentum']}, λ_sparse={best_config['lambda_sparse']:.0e}")
print(f"  max_epochs={MAX_EPOCHS}, patience={PATIENCE}")
print("=" * 70)

# ── Val split for final training ──────────────────────────────────────────────
X_final_tr  = X_train_scaled[n_val_split:]
y_final_tr  = y_train_scaled[n_val_split:]
X_final_val = X_train_scaled[:n_val_split]
y_final_val = y_train_scaled[:n_val_split]

final_model = TabNetRegressor(
    n_d              = best_config['n_d'],
    n_a              = best_config['n_d'],
    n_steps          = best_config['n_steps'],
    momentum         = best_config['momentum'],
    lambda_sparse    = best_config['lambda_sparse'],
    optimizer_params = {'lr': best_config['lr']},   # lr goes in constructor, not fit()
    seed             = RANDOM_SEED,
    device_name      = 'auto',
    verbose          = 1,
)

final_model.fit(
    X_final_tr, y_final_tr,
    eval_set           = [(X_final_val, y_final_val)],
    eval_metric        = ['mse'],
    max_epochs         = MAX_EPOCHS,
    patience           = PATIENCE,
    batch_size         = best_config['batch_size'],
    virtual_batch_size = max(8, best_config['batch_size'] // 4),
    drop_last          = False,
)

best_val_final   = min(final_model.history['val_0_mse'])
best_epoch_final = final_model.history['val_0_mse'].index(best_val_final)
print(f"\n✓ Best val MSE : {best_val_final:.6f}  at epoch {best_epoch_final + 1}")
print(f"  Total epochs  : {len(final_model.history['loss'])}")

# ── Save model ────────────────────────────────────────────────────────────────
model_path = f'{OUTPUT_DIR}/final_tabnet_model'
final_model.save_model(model_path)
print(f"✓ Model saved  : {model_path}.zip")

# ── Loss vs Epoch plot ────────────────────────────────────────────────────────
train_losses = final_model.history['loss']
val_losses   = final_model.history['val_0_mse']
epochs_range = range(1, len(train_losses) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, train_losses, label='Train Loss', color='#3498DB',
        linewidth=2, alpha=0.85)
ax.plot(epochs_range, val_losses,   label='Val Loss',   color='#E74C3C',
        linewidth=2, alpha=0.85)
ax.axvline(best_epoch_final + 1, color='gold', linestyle='--', linewidth=1.8,
           label=f'Best epoch: {best_epoch_final + 1}')
ax.set_xlabel('Epoch', fontsize=13, fontweight='bold')
ax.set_ylabel('MSE Loss', fontsize=13, fontweight='bold')
ax.set_title(
    f'Final Training — Loss vs Epoch\n'
    f'(n_d={best_config["n_d"]}, n_steps={best_config["n_steps"]}, '
    f'lr={best_config["lr"]:.0e}, batch={best_config["batch_size"]})',
    fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/final_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/final_training_loss.png")


In [ ]:
# ── Phase 4: Model Evaluation on Test Set ────────────────────────────────────
print("=" * 70)
print("PHASE 4: MODEL EVALUATION — TEST SET")
print("=" * 70)

# ── Predictions: scaled → log10 → original units ──────────────────────────────
y_pred_scaled = final_model.predict(X_test_scaled)          # numpy (N, output_dim)
y_pred_log    = scaler_y.inverse_transform(y_pred_scaled)   # back to log10 scale
y_true_log    = scaler_y.inverse_transform(y_test_scaled)   # back to log10 scale
y_pred_orig   = 10 ** y_pred_log                            # original units
y_true_orig   = 10 ** y_true_log                            # original units

# ── Residuals for downstream residual analysis ────────────────────────────────
residuals = y_true_log - y_pred_log                         # log10 space

# ── Per-output metrics ────────────────────────────────────────────────────────
r2_per_output   = [r2_score(y_true_log[:, i], y_pred_log[:, i])
                   for i in range(output_dim)]
mae_per_output  = [mean_absolute_error(y_true_log[:, i], y_pred_log[:, i])
                   for i in range(output_dim)]
rmse_per_output = [np.sqrt(mean_squared_error(y_true_log[:, i], y_pred_log[:, i]))
                   for i in range(output_dim)]

# ── Overall metrics ───────────────────────────────────────────────────────────
mae_overall  = mean_absolute_error(y_true_log,  y_pred_log)
mse_overall  = mean_squared_error(y_true_log,   y_pred_log)
rmse_overall = np.sqrt(mse_overall)
r2_overall   = r2_score(y_true_log, y_pred_log)
mape_overall = float(np.mean(np.abs((y_true_orig - y_pred_orig) / (y_true_orig + 1e-12))) * 100)

print(f"\n{'Metric':<28}  {'Value':>12}")
print("-" * 45)
print(f"{'MAE  (log10 units)':<28}  {mae_overall:>12.6f}")
print(f"{'MSE  (log10 units)':<28}  {mse_overall:>12.6f}")
print(f"{'RMSE (log10 units)':<28}  {rmse_overall:>12.6f}")
print(f"{'R²   (overall)':<28}  {r2_overall:>12.6f}")
print(f"{'MAPE (original units, %)':<28}  {mape_overall:>12.4f}")
print("-" * 45)
print(f"\n  Mean R²  across {output_dim} outputs : {np.mean(r2_per_output):.6f}")
print(f"  Min  R²                         : {np.min(r2_per_output):.6f}  "
      f"(output #{np.argmin(r2_per_output)}  →  {output_target_cols[np.argmin(r2_per_output)]})")
print(f"  Max  R²                         : {np.max(r2_per_output):.6f}")
print(f"  Mean MAE across {output_dim} outputs : {np.mean(mae_per_output):.6f}")
print(f"  Mean RMSE                       : {np.mean(rmse_per_output):.6f}")

# ── Plot 1: R² per output bar ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#27AE60' if v >= 0.9 else '#F39C12' if v >= 0.7 else '#E74C3C'
          for v in r2_per_output]
ax.bar(range(output_dim), r2_per_output, color=colors, edgecolor='white', linewidth=0.4)
ax.axhline(0.9, color='green',      linestyle='--', linewidth=1.5, label='R²=0.90')
ax.axhline(0.7, color='darkorange', linestyle='--', linewidth=1.5, label='R²=0.70')
ax.set_xlabel('Output Index (PGA, PGV, SA periods)', fontsize=12, fontweight='bold')
ax.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax.set_title(f'R² per Output — Test Set  (overall R²={r2_overall:.4f})',
             fontsize=13, fontweight='bold')
ax.set_ylim(max(0, min(r2_per_output) - 0.05), 1.02)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eval_r2_per_output.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/eval_r2_per_output.png")

# ── Plot 2: Predicted vs Observed scatter (PGA, PGV, Sa T=1s) ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titles    = ['PGA  (log10 g)', 'PGV  (log10 cm/s)', 'SA T=1.0 s  (log10 g)']
idx_sa1   = next((i for i, c in enumerate(output_target_cols) if 'T1.0' in c), min(output_dim - 1, 30))
for ax, idx, title in zip(axes, [0, 1, idx_sa1], titles):
    lo = min(y_true_log[:, idx].min(), y_pred_log[:, idx].min())
    hi = max(y_true_log[:, idx].max(), y_pred_log[:, idx].max())
    ax.scatter(y_true_log[:, idx], y_pred_log[:, idx],
               s=4, alpha=0.35, color='#2980B9', rasterized=True)
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.8, label='1:1 line')
    ax.set_xlabel('Observed', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted', fontsize=11, fontweight='bold')
    ax.set_title(f'{title}\nR²={r2_per_output[idx]:.4f}  RMSE={rmse_per_output[idx]:.4f}',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
plt.suptitle('Predicted vs Observed — Test Set', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eval_pred_vs_obs.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/eval_pred_vs_obs.png")


In [ ]:
# ── Phase 5: Residual Analysis ────────────────────────────────────────────────
# Residual = log10(Observed) – log10(Predicted)
# Positive → under-prediction  |  Negative → over-prediction
print("=" * 70)
print("PHASE 5: RESIDUAL ANALYSIS")
print("=" * 70)

pga_resid = residuals[:, 0]   # index 0 = log10_PGA

idx_mag  = input_feature_cols.index('Earthquake Magnitude')
idx_rjb  = input_feature_cols.index('Joyner-Boore Dist. (km)')
idx_vs30 = input_feature_cols.index('Vs30 (m/s) selected for analysis')

Mw   = X_test[:, idx_mag ].astype(float)
Rjb  = X_test[:, idx_rjb ].astype(float)
Vs30 = X_test[:, idx_vs30].astype(float)

# ── Equal-count binning ───────────────────────────────────────────────────────
def equal_count_bins(x, y, n_bins=10, log_x=False):
    order  = np.argsort(x)
    xs, ys = x[order], y[order]
    groups = np.array_split(np.arange(len(xs)), n_bins)
    centers, means, stds, counts = [], [], [], []
    for g in groups:
        if len(g) < 2:
            continue
        xg, yg = xs[g], ys[g]
        c = np.exp(np.mean(np.log(np.clip(xg, 1e-9, None)))) if log_x else np.mean(xg)
        centers.append(c)
        means.append(np.mean(yg))
        stds.append(np.std(yg, ddof=1))
        counts.append(len(g))
    return np.array(centers), np.array(means), np.array(stds), np.array(counts)

# ── Generic residual plot ─────────────────────────────────────────────────────
def residual_plot(x, y, xlabel, title, filename,
                  log_x=False, n_bins=10, ylim=(-1.6, 1.6)):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(x, y, s=12, facecolors='none', edgecolors='#22AA22',
               linewidths=0.8, alpha=0.55, rasterized=True, zorder=2, label='Residual')
    ax.axhline(0.0, color='black', linewidth=1.2, linestyle='-', zorder=3)
    bc, bm, bs, _ = equal_count_bins(x, y, n_bins=n_bins, log_x=log_x)
    ax.errorbar(bc, bm, yerr=bs, fmt='D', color='#D13434', ecolor='#060606',
                elinewidth=2.0, capsize=5, capthick=2.0, markersize=9,
                markeredgecolor='#060606', markeredgewidth=0.8,
                linewidth=0, zorder=5, label='Mean ± 1σ')
    if log_x:
        ax.set_xscale('log')
    ax.set_ylim(ylim)
    ax.set_xlabel(xlabel,         fontsize=12, fontweight='bold')
    ax.set_ylabel('Residual',     fontsize=12, fontweight='bold')
    ax.set_title(f'PGA\n{title}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11, framealpha=0.9,
              loc='upper right' if log_x else 'upper left')
    ax.grid(True, which='both' if log_x else 'major',
            linestyle='-', linewidth=0.6, alpha=0.45, color='#CCCCCC')
    ax.set_facecolor('white')
    fig.patch.set_facecolor('white')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{filename}', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {OUTPUT_DIR}/{filename}")

# ── Figure 1 — PGA vs Magnitude ───────────────────────────────────────────────
residual_plot(Mw, pga_resid,
              xlabel='Magnitude',
              title='Inter-Event Residual',
              filename='residuals_PGA_vs_magnitude.png',
              log_x=False, n_bins=10)

# ── Figure 2 — PGA vs Rjb ─────────────────────────────────────────────────────
mask = Rjb > 0.1
residual_plot(Rjb[mask], pga_resid[mask],
              xlabel='Closest distance to Rupture, $R_{rup}$ (km)',
              title='Intra-Event Residual (Measured $V_{S30}$)',
              filename='residuals_PGA_vs_distance.png',
              log_x=False, n_bins=10)

# ── Figure 3 — PGA vs Vs30 ────────────────────────────────────────────────────
residual_plot(Vs30, pga_resid,
              xlabel='Shear wave velocity, $V_{S30}$ (m/s)',
              title='Intra-Event Residual (Measured $V_{S30}$)',
              filename='residuals_PGA_vs_vs30.png',
              log_x=False, n_bins=10)


In [ ]:
# ── Phase 6: Parametric Study — Response Spectra ─────────────────────────────
print("=" * 80)
print("PHASE 6: PARAMETRIC STUDY - RESPONSE SPECTRA")
print("=" * 80)

def predict_spectrum(Mw, Rjb, mechanism, Vs30):
    """Return SA spectrum (g) for the given source/path/site parameters."""
    X_raw    = np.array([[Mw, Rjb, mechanism, Vs30, np.log10(Vs30), np.log10(Rjb)]],
                        dtype='float32')
    X_scaled = scaler_X.transform(X_raw)
    y_scaled = final_model.predict(X_scaled)
    y_log    = scaler_y.inverse_transform(y_scaled)
    y_lin    = 10 ** y_log
    return y_lin[0, 2:]   # skip PGA (idx 0) and PGV (idx 1)

# ── 6.1 Magnitude Variation ───────────────────────────────────────────────────
print("\n" + "=" * 80)
print("PHASE 6.1: MAGNITUDE VARIATION")
print("=" * 80)

fixed_rjb  = 10.0
fixed_vs30 = 760.0
fixed_mech = 1
magnitudes  = [4.0, 5.0, 6.0, 7.0, 7.5]
colors_mag  = ['#E74C3C', '#27AE60', '#3498DB', '#2C3E50', '#E91E63']

print(f"  Fixed: Rjb={fixed_rjb} km, Vs30={fixed_vs30} m/s, Mechanism={fixed_mech}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for mag, color in zip(magnitudes, colors_mag):
    sa = predict_spectrum(mag, fixed_rjb, fixed_mech, fixed_vs30)
    ax1.loglog(sa_periods, sa, '-', linewidth=2.5, label=f'$M_w$ {mag}', color=color)
    ax2.semilogy(sa_periods, sa, '-', linewidth=2.5, label=f'$M_w$ {mag}', color=color)
for ax in (ax1, ax2):
    ax.set_xlabel('Period (s)', fontsize=14, fontweight='bold')
    ax.set_ylabel('PSA (g)',    fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.4, which='both')
    ax.set_xlim(0.01, 5)
    ax.set_ylim(1e-5, 10)
ax1.set_title('Response Spectra (Log-Log Scale)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, loc='lower left', framealpha=0.9)
ax2.set_title(f'Response Spectra – Magnitude Variation\nVs30={fixed_vs30} m/s, Rjb={fixed_rjb} km',
              fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/parametric_magnitude.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/parametric_magnitude.png")

# ── 6.2 Distance Variation ────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("PHASE 6.2: DISTANCE VARIATION")
print("=" * 80)

fixed_mag_dist = 7.0
distances      = [10.0, 22.36, 50.0, 111.80, 250.0]
colors_dist    = ['#E74C3C', '#E67E22', '#F1C40F', '#27AE60', '#3498DB']

print(f"  Fixed: Mw={fixed_mag_dist}, Vs30={fixed_vs30} m/s, Mechanism={fixed_mech}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for rjb, color in zip(distances, colors_dist):
    sa = predict_spectrum(fixed_mag_dist, rjb, fixed_mech, fixed_vs30)
    ax1.loglog(sa_periods, sa, '-', linewidth=2.5, label=f'Rjb={rjb:.2f} km', color=color)
    ax2.semilogy(sa_periods, sa, '-', linewidth=2.5, label=f'Rjb={rjb:.2f} km', color=color)
for ax in (ax1, ax2):
    ax.set_xlabel('Period (s)', fontsize=14, fontweight='bold')
    ax.set_ylabel('SA (g)',     fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.4, which='both')
    ax.set_xlim(0.01, 5)
    ax.set_ylim(1e-5, 10)
ax1.set_title('Response Spectra (Log-Log Scale)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, loc='lower left', framealpha=0.9)
ax2.set_title(f'Response Spectra – Distance Variation\nMw={fixed_mag_dist}, Vs30={fixed_vs30} m/s',
              fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/parametric_distance.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/parametric_distance.png")

# ── 6.3 Vs30 (Site Class) Variation ──────────────────────────────────────────
print("\n" + "=" * 80)
print("PHASE 6.3: SITE CONDITION (Vs30) VARIATION")
print("=" * 80)

fixed_mag_vs30 = 6.5
fixed_rjb_vs30 = 10.0
vs30_values    = [1500, 760, 525, 225, 150]
vs30_labels    = ['A (1500 m/s)', 'B (760 m/s)', 'C (525 m/s)', 'D (225 m/s)', 'E (150 m/s)']
colors_vs30    = ['#2C3E50', '#3498DB', '#27AE60', '#F39C12', '#E74C3C']

print(f"  Fixed: Mw={fixed_mag_vs30}, Rjb={fixed_rjb_vs30} km, Mechanism={fixed_mech}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for vs30, label, color in zip(vs30_values, vs30_labels, colors_vs30):
    sa = predict_spectrum(fixed_mag_vs30, fixed_rjb_vs30, fixed_mech, vs30)
    ax1.loglog(sa_periods, sa, '-', linewidth=2.5, label=label, color=color)
    ax2.semilogy(sa_periods, sa, '-', linewidth=2.5, label=label, color=color)
for ax in (ax1, ax2):
    ax.set_xlabel('Period (s)', fontsize=14, fontweight='bold')
    ax.set_ylabel('SA (g)',     fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.4, which='both')
    ax.set_xlim(0.01, 5)
    ax.set_ylim(1e-5, 10)
ax1.set_title('Response Spectra (Log-Log Scale)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, loc='lower left', framealpha=0.9)
ax2.set_title(f'Response Spectra – Vs30 Variation\nMw={fixed_mag_vs30}, Rjb={fixed_rjb_vs30} km',
              fontsize=14, fontweight='bold')
ax2.legend(fontsize=11, loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/parametric_vs30.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/parametric_vs30.png")

print(f"\n{'='*80}")
print("✓ PHASE 6 COMPLETE: All parametric studies done")
print(f"{'='*80}")


In [ ]:
# ── Phase 7.1: TabNet Native Attention-Based Feature Importance ───────────────
print("=" * 80)
print("PHASE 7.1: TABNET NATIVE ATTENTION FEATURE IMPORTANCE")
print("=" * 80)

# explain() returns:
#   M_explain : (n_samples, n_features)  — aggregate attention across all steps
#   masks     : dict  {step_index: (n_samples, n_features)} — per-step attention
M_explain, masks = final_model.explain(X_test_scaled)

feature_names = [
    '$M_w$',
    '$R_{jb}$ (km)',
    'Mechanism',
    '$V_{s30}$ (m/s)',
    '$\\log_{10}(V_{s30})$',
    '$\\log_{10}(R_{jb})$',
]

# ── Global importance bar ─────────────────────────────────────────────────────
mean_attention  = M_explain.mean(axis=0)          # (n_features,)
importance_pct  = mean_attention / mean_attention.sum() * 100

order              = np.argsort(importance_pct)[::-1]
feat_labels_sorted = [feature_names[i] for i in order]
importance_sorted  = importance_pct[order]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(range(len(feature_names)), importance_sorted,
              color='#E67E22', edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(feature_names)))
ax.set_xticklabels(feat_labels_sorted, fontsize=11)
ax.set_ylabel('Relative Importance (%)', fontsize=12, fontweight='bold')
ax.set_title('TabNet Native Feature Importance\n(Mean attention weight across all test samples)',
             fontsize=13, fontweight='bold')
for bar, val in zip(bars, importance_sorted):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0, importance_sorted[0] * 1.18)
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/tabnet_attention_importance.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/tabnet_attention_importance.png")

print("\n  Feature Ranking (native attention):")
for rank, (name, pct) in enumerate(zip(feat_labels_sorted, importance_sorted), 1):
    print(f"    {rank}. {name:<30s}  {pct:.2f}%")

# ── Per-step attention heatmap ────────────────────────────────────────────────
# masks is a dict: {step_index (int): array (n_samples, n_features)}
n_steps_used = len(masks)
step_means   = np.array([masks[k].mean(axis=0) for k in sorted(masks)])  # (n_steps, n_features)
# normalise each step row to sum to 1 for clean visualisation
step_means_norm = step_means / (step_means.sum(axis=1, keepdims=True) + 1e-12)

fig, ax = plt.subplots(figsize=(9, max(3, n_steps_used * 0.7 + 1)))
im = ax.imshow(step_means_norm, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(feature_names)))
ax.set_xticklabels(feature_names, fontsize=10)
ax.set_yticks(range(n_steps_used))
ax.set_yticklabels([f'Step {i+1}' for i in range(n_steps_used)], fontsize=10)
ax.set_xlabel('Feature', fontsize=12, fontweight='bold')
ax.set_ylabel('Attention Step', fontsize=12, fontweight='bold')
ax.set_title('Per-Step Attention Weights\n(Each row normalised to sum = 1)',
             fontsize=13, fontweight='bold')
for i in range(n_steps_used):
    for j in range(len(feature_names)):
        ax.text(j, i, f'{step_means_norm[i, j]:.2f}',
                ha='center', va='center', fontsize=8,
                color='black' if step_means_norm[i, j] < 0.5 else 'white')
plt.colorbar(im, ax=ax, label='Normalised attention weight')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/tabnet_attention_per_step.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/tabnet_attention_per_step.png")


In [ ]:
# ── Phase 7.2: SHAP Analysis ──────────────────────────────────────────────────
import shap

print("=" * 80)
print("PHASE 7.2: SHAP ANALYSIS & FEATURE IMPORTANCE")
print("=" * 80)
print(f"  shap version: {shap.__version__}")

# ── Background & explain sets ─────────────────────────────────────────────────
np.random.seed(RANDOM_SEED)
bg_idx  = np.random.choice(len(X_train_scaled), 300, replace=False)
exp_idx = np.random.choice(len(X_test_scaled),  500, replace=False)

# Robustly detect the device the model sits on; fall back to global `device`
try:
    model_device = next(final_model.network.parameters()).device
except (StopIteration, AttributeError, TypeError):
    model_device = device   # global device set in cell 2

background = torch.tensor(X_train_scaled[bg_idx], dtype=torch.float32).to(model_device)
X_exp_t    = torch.tensor(X_test_scaled[exp_idx], dtype=torch.float32).to(model_device)
X_exp_np   = X_test_scaled[exp_idx]

print(f"\n  Model device       : {model_device}")
print(f"  Background samples : {len(bg_idx)}")
print(f"  Explain  samples   : {len(exp_idx)}")
print("\n  Computing SHAP values via DeepExplainer …")

# ── Wrapper to extract only the main output (TabNet returns tuple: (output, masks)) ──
class TabNetWrapper(torch.nn.Module):
    def __init__(self, network):
        super().__init__()
        self.network = network
    
    def forward(self, x):
        output, _ = self.network(x)  # Unpack tuple; return only tensor output
        return output

final_model.network.eval()
wrapped = TabNetWrapper(final_model.network)
explainer   = shap.DeepExplainer(wrapped, background)
shap_values = explainer.shap_values(X_exp_t, check_additivity=False)

# Normalise to list: one (n_samples, n_features) array per output
if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_list = [shap_values[:, :, i] for i in range(shap_values.shape[2])]
elif isinstance(shap_values, list):
    shap_list = shap_values
else:
    raise ValueError(f"Unexpected shap_values type: {type(shap_values)}")

print(f"  ✓ SHAP computed — {len(shap_list)} output arrays, each {shap_list[0].shape}")

In [ ]:
# ── Phase 7.2a: SHAP Global Feature Importance bar ───────────────────────────
print("\n" + "=" * 80)
print("PHASE 7.2a: SHAP GLOBAL FEATURE IMPORTANCE")
print("=" * 80)

per_feat       = np.mean([np.abs(sv).mean(axis=0) for sv in shap_list], axis=0)
importance_pct = per_feat / per_feat.sum() * 100

order              = np.argsort(importance_pct)[::-1]
feat_labels_sorted = [feature_names[i] for i in order]
importance_sorted  = importance_pct[order]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(range(len(feature_names)), importance_sorted,
              color='#3498DB', edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(feature_names)))
ax.set_xticklabels(feat_labels_sorted, fontsize=11)
ax.set_ylabel('Relative Importance (%)', fontsize=12, fontweight='bold')
ax.set_title('SHAP Global Feature Importance\n(Mean |SHAP| averaged over all outputs)',
             fontsize=13, fontweight='bold')
for bar, val in zip(bars, importance_sorted):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0, importance_sorted[0] * 1.18)
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/shap_feature_importance.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/shap_feature_importance.png")

print("\n  SHAP Ranking:")
for rank, (name, pct) in enumerate(zip(feat_labels_sorted, importance_sorted), 1):
    print(f"    {rank}. {name:<30s}  {pct:.2f}%")

# ── Phase 7.2b: SHAP Beeswarm plots — PGA, PGV, Sa(0.2s), Sa(1.0s) ───────────
print("\n" + "=" * 80)
print("PHASE 7.2b: SHAP BEESWARM PLOTS")
print("=" * 80)

idx_02s = int(np.argmin(np.abs(sa_periods - 0.2))) + 2   # +2: skip PGA, PGV
idx_10s = int(np.argmin(np.abs(sa_periods - 1.0))) + 2
print(f"  T=0.2s → output index {idx_02s}  (actual T={sa_periods[idx_02s-2]:.3f}s)")
print(f"  T=1.0s → output index {idx_10s}  (actual T={sa_periods[idx_10s-2]:.3f}s)")

panels = [
    (0,       'SHAP Summary — PGA',
              f'{OUTPUT_DIR}/shap_beeswarm_PGA.png'),
    (1,       'SHAP Summary — PGV',
              f'{OUTPUT_DIR}/shap_beeswarm_PGV.png'),
    (idx_02s, f'SHAP Summary — PSA at T={sa_periods[idx_02s-2]:.3f}s',
              f'{OUTPUT_DIR}/shap_beeswarm_SA0.2s.png'),
    (idx_10s, f'SHAP Summary — PSA at T={sa_periods[idx_10s-2]:.3f}s',
              f'{OUTPUT_DIR}/shap_beeswarm_SA1.0s.png'),
]

for out_idx, title, fpath in panels:
    fig, ax = plt.subplots(figsize=(9, 5))
    plt.sca(ax)
    shap.summary_plot(
        shap_list[out_idx],
        X_exp_np,
        feature_names=feature_names,
        plot_type='dot',
        show=False,
        color_bar=True,
    )
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('SHAP value (impact on model output)', fontsize=11)
    ax.tick_params(axis='y', labelsize=11)
    plt.tight_layout()
    plt.savefig(fpath, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"✓ Saved: {fpath}")

print(f"\n{'='*80}")
print("✓ ALL PHASES COMPLETE")
print(f"  Results saved to: {OUTPUT_DIR}/")
print(f"{'='*80}")
